In [4]:
import collections
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from PIL import Image
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

sys.path.insert(0, "..")

from src.config import PROJECT_ROOT



In [5]:

# options are: "Qwen3-VL-2B" - 
MODEL_NAME =  "Qwen3-VL-2B"

#Create output folders
try:
    os.mkdir((PROJECT_ROOT / "outputs" / "inferences").resolve())
except FileExistsError:
    print("Directory already exists.")

#TODO if token not present use NONE
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

data = pd.read_csv(f"{PROJECT_ROOT}/data/specs_data_safe.csv")
print(data.shape)
data.head(1)

Directory already exists.
(5000, 52)


,Unnamed: 0,Respondent,Country,gender,age_group,age_group_2,nationality,city_living,city_living_length,ahi,...,left_year,left_path,right_uuid,right_city,right_image_name,right_image_id,right_lat,right_lon,right_year,right_path
0,0,4,Singapore,female,40-49,40+,Singapore,Singapore,Above 5 years,"SGD$82,000 - SGD$99,999",...,2015.0,data/SPECS/svi/Singapore/76d51909-0a1e-4a6b-b0...,2137f518-7738-4109-bab5-ccfc88bd74d6,Santiago,Image_44,44,-33.455094,-70.675773,2016.0,data/SPECS/svi/Santiago/2137f518-7738-4109-bab...


In [6]:

def build_prompt_blocks(row, do_persona=True):
    
    persona_block = f"""Persona profile:
- Country of residence: {row["Country"]}
- Gender: {row["gender"]}
- Age group: {row["age_group"]}
- Education level: {row["education_level_remapped"]}
- Race/ethnicity: {row["race_ethnicity"]}
- Socioeconomic group: {row["ahi_3_remapped"]}

Personality Big Five (scale: 1 = very low, 5 = very high):
- Extraversion: {row["extraversion"]}
- Agreeableness: {row["agreeableness"]}
- Conscientiousness: {row["conscientiousness"]}
- Neuroticism: {row["neuroticism"]}
- Openness: {row["openness"]}

Adopt the perspective of a person with this profile. Let the profile inform how you interpret the images.
"""

    task_block = """Which place looks safer?

You are shown two street-level images of urban places, Image A and Image B. Their order carries no information.
Judge only from features visible in the images. Compare the visible characteristics of both places.
"""

    tail = """
Answer options, pick one:
- "image_A": the place in Image A looks safer.
- "image_B": the place in Image B looks safer.
- "equal": the two places look equally safe.

Provide an justification of your answer, describe which characteristics influenced your perception of safety for each image.

For each image, give exactly 3 short keywords naming visible features of that image only that affected how safe the place looks.

Return only a valid JSON object with exactly these keys, no Markdown, no code fences, no other text:

{
    "answer": "<image_A | image_B | equal>",
    "keywords_image_A": "<keyword1>, <keyword2>, <keyword3>",
    "keywords_image_B": "<keyword1>, <keyword2>, <keyword3>",
    "justification_image_A": "<visible features only of Image A that affected how safe the place looks>",
    "justification_image_B": "<visible features only of Image B that affected how safe the place looks>"
    
}
"""

    head = (persona_block + "\n" + task_block) if do_persona else task_block
    return head, tail


import re


def _repair_unescaped_quotes(text):
    # The model sometimes emits unescaped double quotes inside string values,
    # e.g. a "Reduce Speed Now" sign -> convert the inner quotes to single quotes.
    pattern = re.compile(
        r'("(?:answer|keywords_image_A|keywords_image_B|justification_image_A|justification_image_B)"'
        r'\s*:\s*")(.*?)("\s*(?:,|\}))',
        re.DOTALL,
    )
    return pattern.sub(lambda m: m.group(1) + m.group(2).replace('"', "'") + m.group(3), text)


def parse_json_response(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        repaired = _repair_unescaped_quotes(text).strip()
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            if not repaired.endswith(('"', "}")):
                repaired += '"'
            if not repaired.endswith("}"):
                repaired += "}"
            return json.loads(repaired)


def resize_image(image_path, max_size=1024):
    image = Image.open(image_path).convert("RGB")
    image.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    return image


In [7]:
# Load Qwen3-VL-2B
# Turing (sm_75) and older lack native bfloat16 -> float16 is faster there.
# Ampere and newer (sm_80+) use native bfloat16.
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    dtype = torch.bfloat16
else:
    dtype = torch.float16
print(f"Using dtype: {dtype}")

if MODEL_NAME == "Qwen3-VL-2B":
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen3-VL-2B-Instruct",
        token=hf_token,
        dtype= dtype,
        device_map="auto"
    )
    
    processor = AutoProcessor.from_pretrained(
        "Qwen/Qwen3-VL-2B-Instruct"
    )

    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None

    print(model.dtype, next(model.parameters()).device)
    #print(model.hf_device_map)         # any "cpu" or "disk" → offload
    print(model.generation_config)

    
    print(collections.Counter(str(p.device) for p in model.parameters()))
    print(collections.Counter(str(b.device) for b in model.buffers()))

Using dtype: torch.float16


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

torch.float16 cuda:0
GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": false,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.0
}

Counter({'cuda:0': 625})
Counter({'cuda:0': 3})


In [8]:
ANSWER_MAP_NORMAL = {
    "image_a": "left",
    "image_b": "right",
    "equal": "equal",
}

ANSWER_MAP_INVERTED = {
    "image_a": "right",
    "image_b": "left",
    "equal": "equal",
}

education_mapping = {
    "college_vocational_diploma": "Vocational college education",
    "other": "Not specified",
    "postgraduate": "Postgraduate education",
    "primary_secondary": "Primary and secondary education",
    "undergraduate": "Undergraduate education",
}

# replace() only modifies the listed entries; every other value is kept as-is.
race_mapping = {
    "Middle Eastern or  North African": "Middle Eastern or North African",
    "A race/ethnicity not listed here": "Not specified",
}

data["education_level_remapped"] = data["education_level_remapped"].replace(education_mapping)
data["race_ethnicity"] = data["race_ethnicity"].replace(race_mapping)

In [10]:
def run_experiments(
    data,
    model,
    processor,
    project_dir,
    model_name,
    *,
    do_persona=True,
    invert_images=False,
    start=0,
    stop=None,
    step=1,
    downsize=True,
    size=768,
    max_new_tokens=512,
    timestamp=None,
):
    if timestamp is None:
        timestamp = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M")

    order = "inverted" if invert_images else "normal"
    condition = "persona" if do_persona else "baseline"
    output_path = (
        project_dir
        / "outputs"
        / "inferences"
        / f"mllm_results_{model_name}_{condition}_{order}_{timestamp}.csv"
    )

    results_df = pd.DataFrame(
        columns=[
            "comparison_id",
            "respondent_id",
            "question",
            "human_answer",
            "model_answer",
            "model_justification_left",
            "model_justification_right",
            "model_keywords_left",
            "model_keywords_right",
            "model_raw_response",
            "model_error",
            "left_image_path",
            "right_image_path",
        ]
    )

    selected = [
        (idx, row)
        for idx, row in data.iterrows()
        if (stop is None or idx < stop)
        and idx >= start
        and (step == 1 or idx % step == 0)
        and row["Question"] == "safe"
    ]
    total = len(selected)
    print(
        f"Run {condition}/{order}: rows={total} | start={start} stop={stop} step={step} | "
        f"downsize={downsize} size={size} | output={output_path.name}"
    )

    for i, (idx, row) in enumerate(selected):
        raw_response = None
        stage = "init"
        model_error = None
        left_image_path = None
        right_image_path = None

        print(f"[{condition}/{order}] inference: {i + 1}/{total} (idx={idx})")

        try:
            stage = "prompt"
            head, tail = build_prompt_blocks(row, do_persona)
            prompt = head + tail

            if i == 0:
                print(prompt)

            stage = "load_images"
            left_image_path = (project_dir / row["left_path"]).resolve()
            right_image_path = (project_dir / row["right_path"]).resolve()

            if downsize is False:
                # Load native image
                left_image = Image.open(left_image_path).convert("RGB")
                right_image = Image.open(right_image_path).convert("RGB")
            else:
                # Load resized image
                left_image = resize_image(left_image_path, size)
                right_image = resize_image(right_image_path, size)

            # image_A = first presented image, image_B = second presented image.
            # invert_images flips only the presentation order. Paths and parsed
            # outputs stay in dataset identity (left/right).
            if invert_images:
                first_image, second_image = right_image, left_image
            else:
                first_image, second_image = left_image, right_image

            stage = "inference"
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": head},
                        {"type": "text", "text": "Image A:"},
                        {"type": "image", "image": first_image},
                        {"type": "text", "text": "Image B:"},
                        {"type": "image", "image": second_image},
                        {"type": "text", "text": tail},
                    ],
                }
            ]
            # Prepare text and images together
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )

            # inputs -> model device
            inputs = inputs.to(model.device)

            t0 = time.perf_counter()
            generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            torch.cuda.synchronize()
            print(
                f"generate: {time.perf_counter() - t0:.3f}s | "
                f"new tokens: {generated_ids.shape[1] - inputs.input_ids.shape[1]}"
            )

            # removing input tokens
            generated_ids_trimmed = [
                output_ids[len(input_ids) :]
                for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
            ]

            # Decode
            output_text = processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
            raw_response = output_text[0]
            print(output_text)

            stage = "parse"
            result = parse_json_response(raw_response)

            answer_map = ANSWER_MAP_INVERTED if invert_images else ANSWER_MAP_NORMAL
            model_answer = answer_map.get(result["answer"].strip().lower(), "invalid")

            # Map per-image outputs back to dataset identity (left/right), so all
            # columns in a row use the same reference frame as human_answer.
            if invert_images:
                justification_left = result["justification_image_B"]
                justification_right = result["justification_image_A"]
                keywords_left = result["keywords_image_B"]
                keywords_right = result["keywords_image_A"]
            else:
                justification_left = result["justification_image_A"]
                justification_right = result["justification_image_B"]
                keywords_left = result["keywords_image_A"]
                keywords_right = result["keywords_image_B"]

            new_row = {
                "comparison_id": idx,
                "respondent_id": row["Respondent"],
                "question": row["Question"],
                "human_answer": row["Score"],
                "model_answer": model_answer,
                "model_justification_left": justification_left,
                "model_justification_right": justification_right,
                "model_keywords_left": keywords_left.strip().lower(),
                "model_keywords_right": keywords_right.strip().lower(),
                "model_raw_response": raw_response,
                "model_error": None,
                "left_image_path": left_image_path,
                "right_image_path": right_image_path,
            }

        except Exception as exc:  # noqa: BLE001
            model_error = f"{stage}: {type(exc).__name__}: {exc}"
            print(f"[{idx}] ERROR ({model_error})")
            new_row = {
                "comparison_id": idx,
                "respondent_id": row["Respondent"],
                "question": row["Question"],
                "human_answer": row["Score"],
                "model_answer": "invalid" if raw_response is not None else "error",
                "model_justification_left": None,
                "model_justification_right": None,
                "model_keywords_left": None,
                "model_keywords_right": None,
                "model_raw_response": raw_response,
                "model_error": model_error,
                "left_image_path": left_image_path,
                "right_image_path": right_image_path,
            }

        results_df.loc[len(results_df)] = new_row
        results_df.to_csv(output_path, index=False)
        print(new_row)

    return results_df


timestamp = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M")
START = 2309
STOP = 2312
STEP = 1  # one row every 5 -> one comparison per respondent/persona
DOWNSIZE = True
SIZE = 768
DO_PERSONA = True


""" results_inverted = run_experiments(
    data, model, processor, PROJECT_ROOT, MODEL_NAME,
    do_persona=DO_PERSONA, invert_images=True,
    start=START, stop=STOP, step=STEP,
    downsize=DOWNSIZE, size=SIZE, timestamp=timestamp,
) """


results_normal = run_experiments(
    data, model, processor, PROJECT_ROOT, MODEL_NAME,
    do_persona=DO_PERSONA, invert_images=False,
    start=START, stop=STOP, step=STEP,
    downsize=DOWNSIZE, size=SIZE, timestamp=timestamp,
)


# RTX 2060 timings
# 13 min - 60 inferencias - fp16 - qwen-2b-vl - imagens reduzidas para 768px - prompt 7
# 137 min - 600 inferencias - fp16 - qwen-2b-vl - imagens reduzidas para 768px - prompt 7

Run persona/normal: rows=3 | start=2309 stop=2312 step=1 | downsize=True size=768 | output=mllm_results_Qwen3-VL-2B_persona_normal_2026-10-02_23-48.csv
[persona/normal] inference: 1/3 (idx=2309)
Persona profile:
- Country of residence: Netherlands
- Gender: male
- Age group: 21-29
- Education level: Postgraduate education
- Race/ethnicity: Black or African American
- Socioeconomic group: middle

Personality Big Five (scale: 1 = very low, 5 = very high):
- Extraversion: 3.0
- Agreeableness: 2.5
- Conscientiousness: 3.5
- Neuroticism: 2.5
- Openness: 3.0

Adopt the perspective of a person with this profile. Let the profile inform how you interpret the images.

Which place looks safer?

You are shown two street-level images of urban places, Image A and Image B. Their order carries no information.
Judge only from features visible in the images. Compare the visible characteristics of both places.

Answer options, pick one:
- "image_A": the place in Image A looks safer.
- "image_B": the plac

In [ ]:
print(model.generation_config)